# 🧠 BrandMind AI — Week 1
## Problem Definition, Requirement Gathering & Dataset EDA
**Tools:** Google Colab · Pandas · Matplotlib · Seaborn · Gemini API

**CRS AI Capstone 2025-26 | Scenario 1**

## 1. Project Scope & Feature List

### Five Integrated Modules:
1. **Logo & Design Studio** — CNN-based logo classification and generation
2. **Creative Content Hub** — NLP tagline/slogan generation via Gemini API
3. **Social Campaign Studio** — ML campaign prediction (CTR, ROI, engagement)
4. **Brand Aesthetics Engine** — KMeans colour palette + font recommendation
5. **Feedback Intelligence** — User feedback loop with model refinement

### Objectives:
- Automatically generate logos aligned to industry and brand tone
- Produce downloadable marketing campaigns with analytics
- Deliver multilingual brand assets for global markets
- Collect feedback to iteratively improve AI outputs

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted successfully')

In [ ]:
# Install dependencies
!pip install pandas matplotlib seaborn google-generativeai -q
print('Dependencies installed')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
print('Libraries imported')

## 2. Dataset Loading & Inspection

In [ ]:
# ── Load all datasets ──
BASE = '/content/drive/MyDrive/BrandMind_Datasets/'

try:
    df_logos     = pd.read_csv(BASE + 'logos.csv')
    df_fonts     = pd.read_csv(BASE + 'fonts.csv')
    df_slogans   = pd.read_csv(BASE + 'slogans.csv')
    df_startups  = pd.read_csv(BASE + 'startups.csv')
    df_marketing = pd.read_csv(BASE + 'marketing.csv')
except FileNotFoundError:
    print('Datasets not found — generating synthetic data for demonstration')
    np.random.seed(42)
    industries = ['Technology','Finance','Healthcare','Retail','Education','Fashion','Travel','Food & Beverage']
    tones = ['Minimalist','Bold','Luxury','Playful','Professional','Creative']
    df_logos     = pd.DataFrame({'image_path': [f'logos/img_{i}.png' for i in range(200)], 'category': np.random.choice(['Abstract','Lettermark','Wordmark','Pictorial','Emblem'], 200), 'industry': np.random.choice(industries, 200), 'style': np.random.choice(tones, 200)})
    df_fonts     = pd.DataFrame({'font_name': [f'Font_{i}' for i in range(100)], 'family': np.random.choice(['serif','sans-serif','script','monospace','display'], 100), 'image_path': [f'fonts/font_{i}.png' for i in range(100)], 'brand_tone': np.random.choice(tones, 100)})
    df_slogans   = pd.DataFrame({'slogan': [f'Sample slogan {i} for brand excellence' for i in range(500)], 'industry': np.random.choice(industries, 500), 'tone': np.random.choice(tones, 500), 'word_count': np.random.randint(3, 12, 500)})
    df_startups  = pd.DataFrame({'company': [f'Startup_{i}' for i in range(300)], 'industry': np.random.choice(industries, 300), 'region': np.random.choice(['North America','Europe','Asia','India','Global'], 300), 'audience': np.random.choice(['B2B','B2C','Enterprise','SMB'], 300), 'tone': np.random.choice(tones, 300), 'revenue_stage': np.random.choice(['Pre-seed','Seed','Series A','Series B'], 300)})
    df_marketing = pd.DataFrame({'platform': np.random.choice(['Instagram','LinkedIn','Twitter','Facebook','TikTok'], 1000), 'campaign_type': np.random.choice(['Image','Video','Carousel','Story'], 1000), 'region': np.random.choice(['North America','Europe','Asia','India'], 1000), 'ctr': np.random.uniform(0.5, 8.5, 1000), 'roi': np.random.uniform(0.8, 6.0, 1000), 'engagement_score': np.random.uniform(1.5, 15.0, 1000), 'budget': np.random.randint(500, 50000, 1000)})
    print('Synthetic datasets created for demonstration')

print('All datasets loaded.')

In [ ]:
# ── Basic Inspection ──
for name, df in [('Logos', df_logos), ('Fonts', df_fonts), ('Slogans', df_slogans), ('Startups', df_startups), ('Marketing', df_marketing)]:
    print(f'\n{'='*50}')
    print(f'Dataset: {name} | Shape: {df.shape}')
    print('--- .head() ---')
    display(df.head(3))
    print('--- .info() ---')
    df.info()
    print('--- .describe() ---')
    display(df.describe(include='all').T.head(8))
    print('--- Missing values ---')
    print(df.isnull().sum())

## 3. Data Quality Checks

In [ ]:
# ── Missing values & duplicates ──
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

datasets = [('Logos', df_logos), ('Slogans', df_slogans), ('Marketing', df_marketing)]
for ax, (name, df) in zip(axes, datasets):
    missing = df.isnull().sum()
    if missing.sum() == 0:
        ax.text(0.5, 0.5, f'{name}\nNo missing values ✓', ha='center', va='center', fontsize=12, transform=ax.transAxes)
        ax.axis('off')
    else:
        missing[missing > 0].plot(kind='bar', ax=ax, color='#b8975a')
        ax.set_title(f'{name} — Missing Values')
        ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig('data_quality_report.png', dpi=150, bbox_inches='tight')
plt.show()

# Duplicates
for name, df in [('Logos', df_logos), ('Fonts', df_fonts), ('Slogans', df_slogans)]:
    dups = df.duplicated().sum()
    print(f'{name}: {dups} duplicates found')

## 4. Dataset Mapping & Notes

| Dataset | Key Columns | Module |
|---------|-------------|--------|
| Logo Dataset | image_path, category, industry, style | Logo Studio (W2), Colour Engine (W5), Animation (W6) |
| Font Dataset | font_name, family, image_path, brand_tone | Font Engine (W3), Animation (W6) |
| Slogan Dataset | slogan, industry, tone, word_count | Content Hub (W4), Multilingual (W8) |
| Startups Dataset | company, industry, region, audience, tone | Persona (W4), Campaign (W7) |
| Marketing Dataset | platform, campaign_type, ctr, roi, engagement_score | Campaign Prediction (W7) |

## 5. Preliminary Visualisations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('BrandMind AI — Preliminary EDA Visualisations', fontsize=14, fontweight='bold')

# 1. Logo categories
df_logos['category'].value_counts().plot(kind='bar', ax=axes[0,0], color='#1a1a18')
axes[0,0].set_title('Logo Category Distribution')
axes[0,0].set_xlabel(''); axes[0,0].tick_params(axis='x', rotation=30)

# 2. Startup industries
df_startups['industry'].value_counts().plot(kind='barh', ax=axes[0,1], color='#b8975a')
axes[0,1].set_title('Startup Industry Distribution')

# 3. Font families
df_fonts['family'].value_counts().plot(kind='pie', ax=axes[0,2], autopct='%1.1f%%', colors=['#1a1a18','#b8975a','#6b6a65','#d4c4a0','#2d4a3e'])
axes[0,2].set_title('Font Family Distribution')

# 4. Slogan word count
sns.histplot(df_slogans['word_count'], ax=axes[1,0], color='#b8975a', kde=True)
axes[1,0].set_title('Slogan Word Count Distribution')

# 5. Marketing platform distribution
df_marketing['platform'].value_counts().plot(kind='bar', ax=axes[1,1], color='#2d4a3e')
axes[1,1].set_title('Campaign Platform Distribution')
axes[1,1].tick_params(axis='x', rotation=30)

# 6. CTR by platform
sns.boxplot(data=df_marketing, x='platform', y='ctr', ax=axes[1,2], palette='muted')
axes[1,2].set_title('CTR Distribution by Platform')
axes[1,2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('eda_visualisations.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Slogan word frequency ──
from collections import Counter
import re

all_words = ' '.join(df_slogans['slogan'].str.lower()).split()
stopwords = {'for','the','a','an','and','or','of','to','in','is','it','our','your','we','you','with','on','by','be'}
word_freq = Counter(w for w in all_words if w not in stopwords and len(w) > 3)

top_words = dict(word_freq.most_common(15))
plt.figure(figsize=(10, 4))
plt.barh(list(top_words.keys())[::-1], list(top_words.values())[::-1], color='#b8975a')
plt.title('Top 15 Slogan Words (excluding stopwords)')
plt.xlabel('Frequency')
plt.tight_layout()
plt.savefig('slogan_word_frequency.png', dpi=150, bbox_inches='tight')
plt.show()

## Week 1 Complete ✅
**Deliverables:**
- ✅ Documented problem statement and 5-module feature map
- ✅ Dataset exploration report with column insights
- ✅ Data quality checks (missing values, duplicates)
- ✅ Preliminary visualisations (logo categories, industries, slogan frequency)
- ✅ Dataset-to-module mapping table

**Next:** Week 2 — CNN Logo Classifier